# Coleta inicial do PIB municipal e VAB por setor

In [ ]:
import csv
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
from openpyxl import load_workbook

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
archive = root / "pib-ibge.zip"
url = "https://ftp.ibge.gov.br/Pib_Municipios/2022_2023/base/base_de_dados_2010_2023_xlsx.zip"
if not archive.exists():
    urlretrieve(url, archive)

fields = ["codigo_municipio", "ano", "municipio", "uf", "pib_mil_reais",
          "vab_agro_mil_reais", "vab_industria_mil_reais", "vab_servicos_mil_reais"]
rows = []
with ZipFile(archive) as bundle:
    name, = [name for name in bundle.namelist() if name.endswith(".xlsx")]
    with bundle.open(name) as source:
        workbook = load_workbook(source, read_only=True, data_only=True)
        for row in workbook["PIB dos Municípios"].iter_rows(min_row=2, values_only=True):
            year, uf = row[0], row[4]
            if uf in {"ES", "MG", "RJ", "SP"}:
                rows.append((str(row[6]), year, row[7], uf, row[38], *row[32:35]))
        workbook.close()

keys = {(row[0], row[1]) for row in rows}
municipalities = {row[0] for row in rows if row[1] == 2023}
assert len(rows) == len(keys), "Chaves duplicadas"
assert len(municipalities) == 1668, "Cobertura inesperada em 2023"
assert all((code, year) in keys for code in municipalities for year in range(2020, 2024))
assert all(isinstance(row[4], (int, float)) and row[4] > 0 for row in rows)

output = root / "data/raw/base_integrada.csv"
output.parent.mkdir(parents=True, exist_ok=True)
with output.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(fields)
    writer.writerows(sorted(rows))
print(len(rows), "linhas;", len(municipalities), "municípios em 2023")
